## 6. Create Feature Store

Code learned and followed from AAI 540 Lab 3

In [1]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.2 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


In [2]:
#!pip install -q PyAthena

In [1]:
import boto3
import sagemaker

original_boto3_version = boto3.__version__
# %pip install 'boto3>1.17.21'

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
from sagemaker.session import Session

region = boto3.Session().region_name

boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

In [3]:
# declare bucket name
#default_s3_bucket_name = feature_store_session.default_bucket()
default_s3_bucket_name = "usdmsaai540-spring2026-team1"

prefix = "sagemaker-featurestore-gasturbine"

print(default_s3_bucket_name)

usdmsaai540-spring2026-team1


In [4]:
from sagemaker import get_execution_role

# call for the role information.
role = get_execution_role()
print(role)

arn:aws:iam::472407119167:role/LabRole


In [5]:
from pyathena import connect

In [6]:
import pandas as pd
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

In [7]:

# Set S3 path to Parquet data
s3_path_parquet = f's3://{bucket}/CCPP/data//parquet'
# Set Athena parameters
# Set Athena parameters
database_name = "ccpp_aws_fp"
table_name_csv = "final_data_csv"
table_name_parquet = "final_data_parquet_v1"

In [8]:
# Set S3 staging directory -- this is a temporary directory used for Athena queries
s3_staging_dir = "s3://{0}/athena/staging".format(bucket)

In [9]:
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [10]:

statement = """SELECT * FROM {}.{}
    ORDER BY ingestion_ts""".format(
    database_name, table_name_parquet
)

print(statement)

SELECT * FROM ccpp_aws_fp.final_data_parquet_v1
    ORDER BY ingestion_ts


In [11]:
ccpp_feature_group_df = pd.read_sql(statement, conn)
ccpp_feature_group_df

/tmp/ipykernel_2898/4051807737.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ccpp_feature_group_df = pd.read_sql(statement, conn)


,seq_id,ingestion_ts,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,1,2026-02-10 02:35:24.729,10.0720,18.132,1027.8,2.6356,346.2,1.2,3.0,1024.7,24.7,1027.3,100.02
1,4,2026-02-10 02:35:24.729,10.0930,18.091,1028.7,2.6372,543.4,2.0,5.0,1023.3,27.8,1027.1,100.07
2,2,2026-02-10 02:35:24.729,10.3860,19.480,1006.5,3.6384,33.8,0.1,0.0,1017.7,24.2,1028.0,100.03
3,5,2026-02-10 02:35:24.729,10.0720,18.232,1026.9,3.8423,0.0,0.0,0.0,1013.7,45.0,1021.9,100.14
4,3,2026-02-10 02:35:24.729,9.9408,17.982,1037.5,2.5758,761.2,2.7,8.0,1013.5,43.4,1017.8,100.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14759,14760,2026-02-10 02:35:24.729,15.0390,36.573,1100.0,4.2813,0.0,0.0,0.0,1033.4,20.9,1038.3,177.49
14760,14761,2026-02-10 02:35:24.729,15.0830,36.533,1100.0,4.3541,0.0,0.0,0.0,1034.1,89.1,1017.1,177.88
14761,14762,2026-02-10 02:35:24.729,15.0290,37.105,1099.9,4.3915,0.0,0.0,0.0,1033.2,87.9,1015.3,177.91
14762,14763,2026-02-10 02:35:24.729,15.0420,36.844,1100.0,4.3749,0.0,0.0,0.0,1034.0,89.1,1015.2,178.31


In [12]:
ccpp_feature_group_df['ingestion_ts'] = ccpp_feature_group_df['ingestion_ts'].astype(str)
ccpp_feature_group_df

,seq_id,ingestion_ts,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,1,2026-02-10 02:35:24.729,10.0720,18.132,1027.8,2.6356,346.2,1.2,3.0,1024.7,24.7,1027.3,100.02
1,4,2026-02-10 02:35:24.729,10.0930,18.091,1028.7,2.6372,543.4,2.0,5.0,1023.3,27.8,1027.1,100.07
2,2,2026-02-10 02:35:24.729,10.3860,19.480,1006.5,3.6384,33.8,0.1,0.0,1017.7,24.2,1028.0,100.03
3,5,2026-02-10 02:35:24.729,10.0720,18.232,1026.9,3.8423,0.0,0.0,0.0,1013.7,45.0,1021.9,100.14
4,3,2026-02-10 02:35:24.729,9.9408,17.982,1037.5,2.5758,761.2,2.7,8.0,1013.5,43.4,1017.8,100.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14759,14760,2026-02-10 02:35:24.729,15.0390,36.573,1100.0,4.2813,0.0,0.0,0.0,1033.4,20.9,1038.3,177.49
14760,14761,2026-02-10 02:35:24.729,15.0830,36.533,1100.0,4.3541,0.0,0.0,0.0,1034.1,89.1,1017.1,177.88
14761,14762,2026-02-10 02:35:24.729,15.0290,37.105,1099.9,4.3915,0.0,0.0,0.0,1033.2,87.9,1015.3,177.91
14762,14763,2026-02-10 02:35:24.729,15.0420,36.844,1100.0,4.3749,0.0,0.0,0.0,1034.0,89.1,1015.2,178.31


### Define FeatureGroups

In [13]:
from time import gmtime, strftime, sleep

ccpp_feature_group_name = "ccpp-feature-group-" + strftime("%d-%H-%M-%S", gmtime())

In [14]:
from sagemaker.feature_store.feature_group import FeatureGroup

ccpp_feature_group = FeatureGroup(
    name=ccpp_feature_group_name, sagemaker_session=feature_store_session
)

In [15]:
import time

current_time_sec = int(round(time.time()))


def cast_object_to_string(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")


# cast object dtype to string. The SageMaker FeatureStore Python SDK will then map the string dtype to String feature type.
cast_object_to_string(ccpp_feature_group_df)

# record identifier and event time feature names
record_identifier_feature_name = "seq_id"
event_time_feature_name = "ingestion_ts"

# append EventTime feature
ccpp_feature_group_df[event_time_feature_name] = pd.Series(
    [current_time_sec] * len(ccpp_feature_group_df), dtype="float64"
)

# load feature definitions to the feature group. SageMaker FeatureStore Python SDK will auto-detect the data schema based on input data.
ccpp_feature_group.load_feature_definitions(data_frame=ccpp_feature_group_df)

[FeatureDefinition(feature_name='seq_id', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='ingestion_ts', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='gt_comp_dis_pressure', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='gt_exhaust_pressure', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='gt_inlet_temp', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='gt_air_filter_diff_pressure', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='solar_radiation', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='solar_energy', feature_type=<FeatureTypeEnum.FR

In [16]:
def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


ccpp_feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=role,
    enable_online_store=True,
)

wait_for_feature_group_creation_complete(feature_group=ccpp_feature_group)

Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
FeatureGroup ccpp-feature-group-10-02-39-42 successfully created.


In [17]:
ccpp_feature_group.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:472407119167:feature-group/ccpp-feature-group-10-02-39-42',
 'FeatureGroupName': 'ccpp-feature-group-10-02-39-42',
 'RecordIdentifierFeatureName': 'seq_id',
 'EventTimeFeatureName': 'ingestion_ts',
 'FeatureDefinitions': [{'FeatureName': 'seq_id', 'FeatureType': 'Integral'},
  {'FeatureName': 'ingestion_ts', 'FeatureType': 'Fractional'},
  {'FeatureName': 'gt_comp_dis_pressure', 'FeatureType': 'Fractional'},
  {'FeatureName': 'gt_exhaust_pressure', 'FeatureType': 'Fractional'},
  {'FeatureName': 'gt_inlet_temp', 'FeatureType': 'Fractional'},
  {'FeatureName': 'gt_air_filter_diff_pressure', 'FeatureType': 'Fractional'},
  {'FeatureName': 'solar_radiation', 'FeatureType': 'Fractional'},
  {'FeatureName': 'solar_energy', 'FeatureType': 'Fractional'},
  {'FeatureName': 'uv_index', 'FeatureType': 'Fractional'},
  {'FeatureName': 'gt_ambient_pressure', 'FeatureType': 'Fractional'},
  {'FeatureName': 'cloud_cover', 'FeatureType': 'Fractional'},

In [18]:
sagemaker_client.list_feature_groups() 

{'FeatureGroupSummaries': [{'FeatureGroupName': 'neighborhood-feature-group-27-05-48-30',
   'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:472407119167:feature-group/neighborhood-feature-group-27-05-48-30',
   'CreationTime': datetime.datetime(2026, 1, 27, 5, 48, 52, 895000, tzinfo=tzlocal()),
   'FeatureGroupStatus': 'Created',
   'OfflineStoreStatus': {'Status': 'Active'}},
  {'FeatureGroupName': 'neighborhood-feature-group-27-05-01-39',
   'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:472407119167:feature-group/neighborhood-feature-group-27-05-01-39',
   'CreationTime': datetime.datetime(2026, 1, 27, 5, 1, 55, 493000, tzinfo=tzlocal()),
   'FeatureGroupStatus': 'Created',
   'OfflineStoreStatus': {'Status': 'Active'}},
  {'FeatureGroupName': 'neighborhood-feature-group-27-04-46-58',
   'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:472407119167:feature-group/neighborhood-feature-group-27-04-46-58',
   'CreationTime': datetime.datetime(2026, 1, 27, 4, 47, 43, 698000, tzinfo=tzlo

### PutRecords into FeatureGroup

In [19]:
ccpp_feature_group.ingest(data_frame=ccpp_feature_group_df, max_workers=3, wait=True)

IngestionManagerPandas(feature_group_name='ccpp-feature-group-10-02-39-42', feature_definitions={'seq_id': {'FeatureName': 'seq_id', 'FeatureType': 'Integral'}, 'ingestion_ts': {'FeatureName': 'ingestion_ts', 'FeatureType': 'Fractional'}, 'gt_comp_dis_pressure': {'FeatureName': 'gt_comp_dis_pressure', 'FeatureType': 'Fractional'}, 'gt_exhaust_pressure': {'FeatureName': 'gt_exhaust_pressure', 'FeatureType': 'Fractional'}, 'gt_inlet_temp': {'FeatureName': 'gt_inlet_temp', 'FeatureType': 'Fractional'}, 'gt_air_filter_diff_pressure': {'FeatureName': 'gt_air_filter_diff_pressure', 'FeatureType': 'Fractional'}, 'solar_radiation': {'FeatureName': 'solar_radiation', 'FeatureType': 'Fractional'}, 'solar_energy': {'FeatureName': 'solar_energy', 'FeatureType': 'Fractional'}, 'uv_index': {'FeatureName': 'uv_index', 'FeatureType': 'Fractional'}, 'gt_ambient_pressure': {'FeatureName': 'gt_ambient_pressure', 'FeatureType': 'Fractional'}, 'cloud_cover': {'FeatureName': 'cloud_cover', 'FeatureType': 'F

In [20]:
record_identifier_value = "10"

featurestore_runtime.get_record(
    FeatureGroupName=ccpp_feature_group.name,
    RecordIdentifierValueAsString=record_identifier_value,
)

{'ResponseMetadata': {'RequestId': '0a9aed09-5d8b-4fc5-8398-22c3ecbd5c23',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '0a9aed09-5d8b-4fc5-8398-22c3ecbd5c23',
   'content-type': 'application/json',
   'content-length': '1113',
   'date': 'Tue, 10 Feb 2026 02:42:29 GMT'},
  'RetryAttempts': 0},
 'Record': [{'FeatureName': 'seq_id', 'ValueAsString': '10'},
  {'FeatureName': 'ingestion_ts', 'ValueAsString': '1770691194.0'},
  {'FeatureName': 'gt_comp_dis_pressure', 'ValueAsString': '10.214'},
  {'FeatureName': 'gt_exhaust_pressure', 'ValueAsString': '18.011'},
  {'FeatureName': 'gt_inlet_temp', 'ValueAsString': '1022.6'},
  {'FeatureName': 'gt_air_filter_diff_pressure', 'ValueAsString': '2.5765'},
  {'FeatureName': 'solar_radiation', 'ValueAsString': '30.5'},
  {'FeatureName': 'solar_energy', 'ValueAsString': '0.1'},
  {'FeatureName': 'uv_index', 'ValueAsString': '0.0'},
  {'FeatureName': 'gt_ambient_pressure', 'ValueAsString': '1021.3'},
  {'FeatureName': 'cloud_cover'

In [21]:
print(ccpp_feature_group.as_hive_ddl())

CREATE EXTERNAL TABLE IF NOT EXISTS sagemaker_featurestore.ccpp-feature-group-10-02-39-42 (
  seq_id INT
  ingestion_ts FLOAT
  gt_comp_dis_pressure FLOAT
  gt_exhaust_pressure FLOAT
  gt_inlet_temp FLOAT
  gt_air_filter_diff_pressure FLOAT
  solar_radiation FLOAT
  solar_energy FLOAT
  uv_index FLOAT
  gt_ambient_pressure FLOAT
  cloud_cover FLOAT
  sea_level_pressure FLOAT
  gt_energy_yield FLOAT
  write_time TIMESTAMP
  event_time TIMESTAMP
  is_deleted BOOLEAN
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe'
  STORED AS
  INPUTFORMAT 'parquet.hive.DeprecatedParquetInputFormat'
  OUTPUTFORMAT 'parquet.hive.DeprecatedParquetOutputFormat'
LOCATION 's3://usdmsaai540-spring2026-team1/sagemaker-featurestore-gasturbine/472407119167/sagemaker/us-east-1/offline-store/ccpp-feature-group-10-02-39-42-1770691204/data'


In [22]:
s3_client = boto3.client("s3", region_name=region)

In [23]:
account_id = boto3.client("sts").get_caller_identity()["Account"]
print(account_id)

ccpp_feature_group_resolved_output_s3_uri = (
    ccpp_feature_group.describe()
    .get("OfflineStoreConfig")
    .get("S3StorageConfig")
    .get("ResolvedOutputS3Uri")
)

ccpp_feature_group_s3_prefix =ccpp_feature_group_resolved_output_s3_uri.replace(
    f"s3://{default_s3_bucket_name}/", ""
)

offline_store_contents = None
while offline_store_contents is None:
    objects_in_bucket = s3_client.list_objects(
        Bucket=default_s3_bucket_name, Prefix=ccpp_feature_group_s3_prefix
    )
    if "Contents" in objects_in_bucket and len(objects_in_bucket["Contents"]) > 1:
        offline_store_contents = objects_in_bucket["Contents"]
    else:
        print("Waiting for data in offline store...\n")
        sleep(60)

print("Data available.")

472407119167
Waiting for data in offline store...

Waiting for data in offline store...

Waiting for data in offline store...

Waiting for data in offline store...

Data available.


In [24]:
ccpp_query = ccpp_feature_group.athena_query()

ccpp_feature_table = ccpp_query.table_name

In [25]:
query_string = f'''
SELECT *
FROM 
    "{ccpp_feature_table}"
'''

print("Running " + query_string)

Running 
SELECT *
FROM 
    "ccpp_feature_group_10_02_39_42_1770691204"



In [26]:
# run Athena query. The output is loaded to a Pandas dataframe.
# dataset = pd.DataFrame()
ccpp__query.run(
    query_string=query_string,
    output_location="s3://" + default_s3_bucket_name + "/" + prefix + "/query_results/",
)
ccpp__query.wait()
dataset = ccpp__query.as_dataframe()

dataset

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│    1 # run Athena query. The output is loaded to a Pandas dataframe.                             │
│    2 # dataset = pd.DataFrame()                                                                  │
│ ❱  3 ccpp__query.run(                                                                            │
│    4 │   query_string=query_string,                                                              │
│    5 │   output_location="s3://" + default_s3_bucket_name + "/" + prefix + "/query_results/",    │
│    6 )                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'ccpp__query' is not defined

In [39]:
ccpp_feature_group.delete()

In [40]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>